In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import pyfar as pf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Audio, display
from algorithms.avendano import ambiance_index, panning_index

%load_ext jupyter_tikz

In [ ]:
def to_dat(X, Y):
    return '\n'.join([f"{x} {y}" for x,y in zip(X,Y)])

In [ ]:
hrirs, sources = pf.signals.files.head_related_impulse_responses(position='horizontal', diffuse_field_compensation=True)

In [ ]:
elevation = 0
azimuth_L = np.pi/3 # 60 degrees left
azimuth_R = -np.pi/3 # 60 degrees right
radius = 1

desired_direction_L = pf.Coordinates.from_spherical_elevation( azimuth_L, elevation, radius)
desired_direction_R = pf.Coordinates.from_spherical_elevation( azimuth_R, elevation, radius)

index_L, _ = sources.find_nearest(desired_direction_L)
index_R, _ = sources.find_nearest(desired_direction_R)

sources.show([index_L, index_R])
plt.show()


In [ ]:
ax = pf.plot.freq_group_delay(hrirs[index_R], label=["Left ear", "Right ear"])
# ax[0].legend()
# ax[1].legend()
plt.show()

In [ ]:
kick = pf.io.read_audio(Path.home() / "Music/freesound/581451__johnnydekk__fractanimal_acoustic_drum_kit_kick_2.wav")
hihat = pf.io.read_audio(Path.home() / "Music/freesound/581473__johnnydekk__fractanimal_acoustic_drum_kit_high_hat_3.wav")
# ax_kick = pf.plot.time_freq(kick)
ax_hihat = pf.plot.time_freq(hihat)
plt.show()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(8,4), sharex=True, layout='tight')



def render(audio, hrir, ax, slug):
    binaural_audio = pf.dsp.convolve(audio, hrir)
    cropped = pf.dsp.time_crop(binaural_audio, [0,0.03], unit='s')
    cropped.time[0] += 1
    cropped.time[1] -= 1
    X = (cropped.times * 1000)

    print("ITD", X[np.argmax(cropped.time, axis=1)])
    print("MAX", np.max(cropped.time, axis=1))
    print("MIN", np.min(cropped.time, axis=1))

    Path(f"../../papers/kappa/figures/{slug}_L.dat").write_text(to_dat(X, cropped.time[0]))
    Path(f"../../papers/kappa/figures/{slug}_R.dat").write_text(to_dat(X, cropped.time[1]))
    # Yl = '\n'.join([f"{y:.3f}" for y in cropped.time[0]])

    pf.plot.time(cropped, label=["Left ear", "Right ear"], ax=ax)
    ax.set_ylim(-2,2)

# def latex_plot_hihat():
    # txt = '\\begin{tikzpicture}'

    # txt += '\\end{tikzpicture}'

    

render(hihat, hrirs[index_R], ax[0], "hihat")
render(kick, hrirs[index_R], ax[1], "kick")
# plt.show()

In [ ]:
music = pf.io.read_audio("/Users/richard/Music/archive/09. Tsubasao Kudasai.flac", stop=int(10. * 44100))
pf.plot.spectrogram(music, freq_scale='log')
plt.show()

In [ ]:
from matplotlib import colormaps, color_sequences

def amplitude_pan(sig: pf.Signal, pan: float) -> pf.Signal:
    assert -1 <= pan <= 1.
    assert sig.cshape == (2,)
    a = (pan + 1) / 2
    al, ar = np.cos(a * 0.5 * np.pi), np.sin(a * 0.5 * np.pi)
    print(a, al, ar)
    left, right = sig.time[0] * al, sig.time[1] * ar
    return pf.Signal(np.stack([left, right]), sig.sampling_rate)

def downmix(sig: pf.Signal) -> pf.Signal:
    assert sig.cshape == (2,)
    a = 1/np.sqrt(2)
    data = np.sum(sig.time, axis=0) * a
    return pf.Signal(np.stack([data, data]), sig.sampling_rate)

def downmix_to_mono(sig: pf.Signal) -> pf.Signal:
    assert sig.cshape == (2,)
    a = 1/np.sqrt(2)
    data = np.sum(sig.time, axis=0) * a
    return pf.Signal(data, sig.sampling_rate)


def make_panned_sound(pan: float = 0.5):
    fvocals = "/Users/richard/Music/freesound/245325__vocalpercussion__vocal_percussion_deh_deh_boom.wav"
    fguitar = "/Users/richard/Music/freesound/474398__bangcorrupt__blues_guitar_intro.flac"
    ftrumpet = "/Users/richard/Music/freesound/2323__synapse__trumpetloop01.aiff"

    def _load(f):
        audio = pf.io.read_audio(f)
        if audio.sampling_rate != 48000:
            audio = pf.dsp.resample(audio, sampling_rate=48000)
        audio = pf.Signal(audio.time[0, :480000], sampling_rate=audio.sampling_rate)
        audio = pf.dsp.normalize(audio, reference_method='energy', target=0.5)
        return audio
    
    vocals = _load(fvocals)
    guitar = _load(fguitar)
    trumpet = _load(ftrumpet)
    # sound1 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f1), sampling_rate=48000), [0.,10.], unit='s')), -pan)
    # sound2 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f2), sampling_rate=48000), [0.,10.], unit='s')), pan)
    # sound = sound1 + sound2
    print(vocals, guitar, trumpet)
    # left = vocals * 0.5 + guitar * 0.7 + trumpet * 0.1
    # right = vocals * 0.5 + guitar * 0.3 + trumpet * 0.9
    left = guitar * 0.7 + trumpet * 0.1
    right = guitar * 0.3 + trumpet * 0.9
    sound = pf.Signal(np.stack([left.time[0], right.time[0]]), 48000)
    return sound

def get_music():
    # f = "/Users/richard/Music/archive/09. Tsubasao Kudasai.flac"
    # return pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f), sampling_rate=48000), [0,480000])
    f = "/Users/richard/Music/projects/panning_and_reverb/panning_and_reverb_wav.wav"
    return pf.io.read_audio(f)

sound = get_music()
# sound = make_panned_sound(pan=0.99)

freqs, times, spect = pf.dsp.spectrogram(sound)
spect = np.swapaxes(spect, 1, 2)


mosaic = """
a
b
b
c
c
d
d
"""
fig, ax = plt.subplot_mosaic(mosaic, figsize=(10,20))

ai = ambiance_index(spect, lamb=0.85)
pi = panning_index(spect)


def decibels(x, mindb=-70.):
    epsilon = np.nextafter(np.float64(0), np.float64(1))
    # epsilon = 1e-16
    return np.clip(20 * np.log10(x + epsilon), a_min=mindb, a_max=float("inf"))

def make_peaks(sound, pps):
    n_sections = int((sound.n_samples * pps) / sound.sampling_rate)
    channel_peaks = []
    for channel in sound.time:
        chunks = np.array_split(channel, n_sections)
        chunks[-2] = np.append(chunks[-2], chunks[-1])
        del chunks[-1]
        channel_peaks.append([np.max(np.abs(chunk)) for chunk in chunks])
    return pf.Signal(np.stack(channel_peaks), pps)

def plot_stereo(sound: pf.Signal, pps: int, ax: plt.Axes, colormap: str = 'berlin'):
    peaks = make_peaks(sound, pps)
    offset = np.max(peaks.time)
    x = np.arange(peaks.n_samples) / peaks.sampling_rate
    color_l, color_r = colormaps.get(colormap)([0.1,0.9])
    ax.fill_between(x, peaks.time[0] + offset, -peaks.time[0] + offset, color=color_l)
    ax.fill_between(x, peaks.time[1] - offset, -peaks.time[1] - offset, color=color_r)
    ax.set_yticks([offset, -offset], labels=["left", "right"])
    ax.set_xlim(x[[0,-1]])

time_range = 10000000

weights = np.sum(spect, axis=0)

panogram = np.vstack([np.histogram(pi[i], bins=100, range=[-1,1], weights=weights[i])[0] for i in range(len(pi))])

# waveform_colours = colormaps.get('berlin')([0.1,0.9])
# ax['a'].set_prop_cycle(color=waveform_colours)
# pf.plot.time(sound, ax=ax['a'], linewidth=0.3, alpha=0.2)
# ax['a'].set_ylim(-0.35, 0.35)

plot_stereo(sound, 400, ax['a'])

extent = (times[0], times[-1], freqs[0], freqs[-1])
ax['b'].imshow(decibels(np.rot90(ai[:time_range])), aspect='auto', cmap='grey', extent=extent)
ax['c'].imshow(np.rot90(pi[:time_range]), aspect='auto', cmap='berlin', extent=extent)
ax['d'].imshow(decibels(np.rot90(panogram[:time_range])), aspect='auto', cmap='gray', extent=extent)

mean_ambience_over_time = np.mean(ai * weights, axis=1)
# ax['e'].plot(mean_ambience_over_time)
# ax[4].plot(decibels(np.mean(panogram, axis=0)))

fig.savefig("waveforms.pdf")

Audio(sound.time, rate=sound.sampling_rate)

In [ ]:
new_length_400pps = int((sound.n_samples * 400) / sound.sampling_rate)
new_length_10pps = int((sound.n_samples * 10) / sound.sampling_rate)
new_length_1pps = int(sound.n_samples / sound.sampling_rate)

print(sound.n_samples, new_length_400pps, new_length_10pps, new_length_1pps)

def make_peaks(sound, pps):
    n_sections = int((sound.n_samples * pps) / sound.sampling_rate)
    channel_peaks = []
    for channel in sound.time:
        chunks = np.array_split(channel, n_sections)
        chunks[-2] = np.append(chunks[-2], chunks[-1])
        del chunks[-1]
        channel_peaks.append([np.max(np.abs(chunk)) for chunk in chunks])
    return pf.Signal(np.stack(channel_peaks), pps)

def plot_stereo(sound: pf.Signal, pps: int, ax: plt.Axes, colormap: str = 'berlin'):
    peaks = make_peaks(sound, pps)
    offset = np.max(peaks.time)
    x = np.arange(peaks.n_samples) / peaks.sampling_rate
    color_l, color_r = colormaps.get(colormap)([0.1,0.9])
    ax.fill_between(x, peaks.time[0] + offset, -peaks.time[0] + offset, color=color_l)
    ax.fill_between(x, peaks.time[1] - offset, -peaks.time[1] - offset, color=color_r)
    ax.set_yticks([])


fig, ax = plt.subplots(3,1, figsize=(10, 10))

plot_stereo(sound, 400, ax[0])
plot_stereo(sound, 10, ax[1])
plot_stereo(sound, 1, ax[2])



In [ ]:
nx, ny = (3,4)
idx = np.mgrid[:nx,:ny].reshape(2, -1).T
frame = [64,256,1024,4096]
hop = [0.5, 0.75, 0.875]


s = 3
fig, ax = plt.subplots(ny, nx, figsize=(nx*s, ny*s), sharex=True)

# file = "659528__johaynes__jos-genedrums-150bpm-42.wav"
# file = "581473__johnnydekk__fractanimal_acoustic_drum_kit_high_hat_3.wav"
file = "245325__vocalpercussion__vocal_percussion_deh_deh_boom.wav"
transient_signal = downmix_to_mono(pf.io.read_audio(f"/Users/richard/Music/freesound/{file}"))
transient_signal = pf.dsp.time_crop(transient_signal, [0.85,1.1], unit='s')

print(transient_signal)
for i,j in idx:
    f = frame[j]
    h = hop[i]
    a = ax[j,i]
    pf.plot.spectrogram(transient_signal, window_length=f, window_overlap_fct=h, ax=a, colorbar=False)
    a.set_title(f"{f}, {int(f*(1-h))}")
    a.set_xlabel(None)
    a.set_ylabel(None)

fig.savefig("spectrogram_accuracy.pdf")

In [ ]:
import librosa
audio, sr = librosa.load(librosa.util.example('trumpet'))
fig, ax = plt.subplots(2,1)

D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
S = librosa.feature.melspectrogram(S=D, sr=sr, n_mels=32)

librosa.display.specshow(data=D, sr=sr, ax=ax[0], y_axis='linear')
librosa.display.specshow(S, sr=sr, ax=ax[1], y_axis='mel')

fig.savefig("mel.pdf")


In [ ]:
import librosa
from python_auditory_toolbox import auditory_toolbox as pat
import scipy

fig, ax = plt.subplots(3,1, figsize=(8, 8), sharex=True)

# audio, sr = librosa.load(librosa.util.example('trumpet', hq=True))
# signal = pf.signals.exponential_sweep_time(48000, [20,20000], sampling_rate=48000)
signal = pf.io.read_audio("/Users/richard/Music/freesound/31912__acclivity__globalwarming_marg.wav")
audio, sr = signal.time[0], signal.sampling_rate

start, end = 0.8, 2.6
audio = audio[int(start * sr):int(end * sr)]

num_bands = 32

coefs = pat.MakeErbFilters(sr, num_bands, low_freq=100)
y = pat.ErbFilterBank(audio, coefs)
y *= np.sign(y) # half wave rectify
y = np.sqrt(y) # square root

# low pass
fc = 800
b, a = scipy.signal.butter(4, fc/sr, btype='low', analog=False)
y = scipy.signal.filtfilt(b, a, y)

chunk_size = int(0.02 * sr)
hop_size = int(0.5 * chunk_size)
chunks = np.lib.stride_tricks.sliding_window_view(y, chunk_size, axis=1)[:, ::hop_size, :]
print(chunks.shape)
# coch = np.sqrt(np.sum(np.pow(chunks, 2), axis=(2,))) # RMS
coch = np.mean(chunks, axis=2)


D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
S = librosa.feature.melspectrogram(S=D, sr=sr, n_mels=num_bands)
print(D.shape, S.shape, coch.shape)

librosa.display.specshow(data=D, sr=sr, ax=ax[0], y_axis='linear')
# librosa.display.specshow(S, sr=sr, ax=ax[1], y_axis='mel')
xmax = D.shape[1] - 1
ax[1].imshow(np.flipud(S), aspect='auto', cmap='magma', extent=(0, xmax, 0, num_bands))
ax[2].imshow(librosa.amplitude_to_db(coch), aspect='auto', cmap='magma', extent=(0, xmax, 0, num_bands))

for i in range(1,3):
    ax[i].set_ylabel("Frequency band")

# y += np.atleast_2d(np.arange(24, 0, -1)).T
# plt.plot(y.T, color='k', linewidth=1)

fig.savefig("spect_comp.pdf")

plt.show()

In [ ]:
mel_filters = librosa.filters.mel(sr=48000, n_fft=4096, n_mels=8, fmax=24000)
mel_filters[mel_filters == 0] = np.nan # showing all the zero weights looks cluttered, so just get rid of them

# mel_freqs = librosa.filters.mel_frequencies(n_mels=8, fmax=24000)
fig, ax = plt.subplots(1,2, figsize=(8, 3), sharey=True)
ax[0].plot(mel_filters.T, color='k')
ax[0].set_xscale('log')
ax[0].set_ylabel("Amplitude")
ax[0].set_xlabel("Frequency (Hz)")
ax[0].set_title("Mel filters (logarithmic)")
ax[1].plot(mel_filters.T, color='k')
ax[1].set_xscale('linear')
# ax[1].set_ylabel("Amplitude")
ax[1].set_xlabel("Frequency (Hz)")
ax[1].set_title("Mel filters (linear)")
fig.tight_layout()
fig.savefig("mel_filters.pdf")

In [ ]:
import scipy
a = 1.99
rv = scipy.stats.gamma(a)
x = np.linspace(0, 10, 200)
y = rv.pdf(x) * np.cos(8*x) - 0.5
y2 = rv.pdf(x-1) * np.cos(8*(x-1)) + 0.5
y3 = (y*0.5) + 0.75
plt.plot(x, y)
plt.plot(x, y2)
plt.plot(x, y3)

def write_path(x, y):
    coords = " ".join([f"({a[0]:.3f},{a[1]:.3f})" for a in  zip(x,y)])
    return f"\\addplot [domain=0:10, no markers, smooth] coordinates {{{coords}}};"

Path("itd_R.tex").write_text(write_path(x,y))
Path("itd_L.tex").write_text(write_path(x,y2))
Path("ild_R.tex").write_text(write_path(x,y))
Path("ild_L.tex").write_text(write_path(x,y3))

print(max(rv.pdf(x)))


In [ ]:
def plot_hrir():
    elevation = 0
    azimuth_L = np.pi/6
    azimuth_R = -azimuth_L
    radius = 1

    desired_direction_L = pf.Coordinates.from_spherical_elevation( azimuth_L, elevation, radius)
    desired_direction_R = pf.Coordinates.from_spherical_elevation( azimuth_R, elevation, radius)
    index_L, _ = sources.find_nearest(desired_direction_L)
    index_R, _ = sources.find_nearest(desired_direction_R)
    print(desired_direction_L.azimuth, desired_direction_R.azimuth)


    dur = 64
    start = 10
    sig = pf.dsp.time_crop(hrirs[index_R], [start,start+dur])
    pf.plot.time(sig)

    X = np.arange(dur)
    Y = sig.time

    def txt(Y, name):
        points = "--".join([f"({x:.3f},{y:.3f})" for x,y in zip(X,Y)])
        return f"\\path[local bounding box={name}, draw=black] {points};"
    
    # print(txt(Y[0]))
    # print(txt(Y[1]))

    Path("../../papers/kappa/figures/brir_L.tex").write_text(txt(Y[0], "left_brir"))
    Path("../../papers/kappa/figures/brir_R.tex").write_text(txt(Y[1], "right_brir"))

plot_hrir()

In [ ]:
# def spiral(theta, a, b):
#     r = a * np.exp(b * theta)
#     x = np.cos(r)
#     y = np.sin(r)
#     return x, y

def cart2pol(xy):
    rho = np.sqrt(xy.real**2 + xy.imag**2)
    phi = np.arctan2(xy.imag, xy.real)
    return np.array([rho, phi])

def pol2cart(rp):
    rho, phi = rp
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return x + y * 1j

def spiral(a, b, t):
    print(t.shape)
    spir = a * np.exp((b + 1j) * t)
    spir.real *= -1
    return spir
    # rho, phi = cart2pol(spir)
    # # phi += np.pi/2
    # return pol2cart([rho, phi])

def coord(xy):
    return f"({xy.real:.3f},{xy.imag:.3f})"

def plot_cochlea(t, a=1, b=0.08, tension=0.618):
    d = 0.18
    inner = spiral(a, b, t)
    outer1 = spiral(a * (1+d), b, t)
    outer2 = spiral(a * (1-d), b, t)

    freqs = [200, 400, 800, 1600, 3200, 6400, 12000, 20000]
    log_pos = np.log10(freqs)
    log_pos -= min(log_pos)
    log_pos /= max(log_pos)
    log_pos_t = np.interp(log_pos, np.linspace(-0.01, 1.01, len(t)), t)

    base_label = spiral(a, b, t[-1:] + 0.168)[0]
    apex_label = inner[0] + 0.618 - 0.2j

    basilar_membrane_label1 = spiral(a, b, t[-4:-3])[0]
    basilar_membrane_label2 = pol2cart(cart2pol(basilar_membrane_label1) + [1.618, 0])

    frq_marks = spiral(a, b, log_pos_t)

    def path(XY, colour):
        points = " ".join([f"({x:.3f},{y:.3f})" for x,y in zip(XY.real, XY.imag)])
        return f"\\draw[{colour}] plot [smooth, tension={tension}] coordinates {{ {points} }};"

    def txt():
        txt = "\\begin{tikzpicture}"
        txt += path(inner, "black, dashed")
        txt += path(outer1, "black, thick")
        txt += path(outer2, "black, thick")
        txt += f"\\draw[black, thick] {coord(outer1[0])} to[bend right=90, looseness=1.618] {coord(outer2[0])};"
        txt += f"\\draw[black, thick] {coord(outer1[-1])} to[bend left=90, looseness=1] {coord(inner[-1])};"
        txt += f"\\draw[black, thick] {coord(outer2[-1])} to[bend right=90, looseness=1] {coord(inner[-1])};"

        txt += f"\\node[] (base_label) at {coord(base_label)} {{Base}};"
        txt += f"\\node[] (apex_label) at {coord(apex_label)} {{Apex}};"
        txt += f"\\draw[latex-] {coord(basilar_membrane_label1)} -- {coord(basilar_membrane_label2)};"
        txt += f"\\node[anchor=north] (basilar_membrane_label) at {coord(basilar_membrane_label2)} {{Basilar membrane}};"
        for pos,frq in zip(frq_marks, freqs):
            txt += f"\\node[circle, fill=black, draw=black, minimum size=4pt, text width=0pt, inner sep=0] (point_{frq}) at {coord(pos)} {{}};"
            txt += f"\\node[anchor=south, fill=white, draw=black, outer sep=4pt] (point_{frq}) at {coord(pos)} {{{frq}}};"

        txt += "\\end{tikzpicture}"
        return txt.replace('\\', '\n\\')

    return txt()

drawing = plot_cochlea(np.linspace(0, 18, 50))
print(drawing)

%tikz "$drawing"

    
